In [2]:
# import libraries 
import pandas as pd
import matplotlib.pyplot as plt
import os 
import math
import sys
from pathlib import Path
import seaborn as sns

from src.data.process_data import *
from src.data.clean_data import *

# Add 'src' to the system path
sys.path.append(str(Path().resolve() / 'src'))

In [3]:
df_movies = pd.read_csv('data/processed/movies.csv')
df_characters = pd.read_csv('data/processed/characters.csv')
df_plots = pd.read_csv('data/processed/plot_summaries.csv')
df_tmdb = pd.read_csv('data/TMDB_movie_dataset_v11.csv')
df_tmdb = df_tmdb[['title', 'vote_average', 'vote_count', 'release_date',
       'revenue', 'runtime', 'budget', 'overview',
       'popularity', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords']]
df_movies_summaries = pd.merge(df_movies, df_plots, on='Wikipedia movie ID')

In [14]:
df_tmdb[(df_tmdb['genres'] is not None) & (df_tmdb['genres'].str.contains("War"))].info()

<class 'pandas.core.frame.DataFrame'>
Index: 10441 entries, 30 to 1125710
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 10441 non-null  object 
 1   vote_average          10441 non-null  float64
 2   vote_count            10441 non-null  int64  
 3   release_date          10063 non-null  object 
 4   revenue               10441 non-null  int64  
 5   runtime               10441 non-null  int64  
 6   budget                10441 non-null  int64  
 7   overview              8882 non-null   object 
 8   popularity            10441 non-null  float64
 9   genres                10441 non-null  object 
 10  production_companies  7858 non-null   object 
 11  production_countries  8841 non-null   object 
 12  spoken_languages      8868 non-null   object 
 13  keywords              4937 non-null   object 
dtypes: float64(2), int64(4), object(8)
memory usage: 1.2+ MB


In [15]:
df_movies_summaries.head()

,Wikipedia movie ID,Freebase ID,Movie name,Release date,Box office revenue,Runtime,Languages,Countries,Genres,Summary
0,9363483,/m/0285_cd,White Of The Eye,1987,NaN,110.0,"{""/m/02h40lc"": ""English Language""}","{""/m/07ssc"": ""United Kingdom""}","{""/m/01jfsb"": ""Thriller"", ""/m/0glj9q"": ""Erotic...",A series of murders of rich young women throug...
1,261236,/m/01mrr1,A Woman in Flames,1983,NaN,106.0,"{""/m/04306rv"": ""German Language""}","{""/m/0345h"": ""Germany""}","{""/m/07s9rl0"": ""Drama""}","Eva, an upper class housewife, becomes frustra..."
2,18998739,/m/04jcqvw,The Sorcerer's Apprentice,2002,NaN,86.0,"{""/m/02h40lc"": ""English Language""}","{""/m/0hzlz"": ""South Africa""}","{""/m/0hqxf"": ""Family Film"", ""/m/01hmnh"": ""Fant...","Every hundred years, the evil Morgana returns..."
3,6631279,/m/0gffwj,Little city,1997-04-04,NaN,93.0,"{""/m/02h40lc"": ""English Language""}","{""/m/09c7w0"": ""United States of America""}","{""/m/06cvj"": ""Romantic comedy"", ""/m/0hj3n0w"": ...","Adam, a San Francisco-based artist who works a..."
4,171005,/m/016ywb,Henry V,1989-11-08,10161099.0,137.0,"{""/m/02h40lc"": ""English Language""}","{""/m/07ssc"": ""United Kingdom""}","{""/m/04xvh5"": ""Costume drama"", ""/m/082gq"": ""Wa...",{{Plot|dateAct 1Act 2Act 3Act 4Act 5 Finally n...


In [16]:
df_movies_summaries.drop(['Wikipedia movie ID', 'Freebase ID'], axis=1, inplace=True)

In [17]:
df_movies_summaries['Genres'] = df_movies_summaries["Genres"].apply(extract_tuples_values)


In [18]:
df_movies_summaries['Genres'] = df_movies_summaries['Genres'].apply(lambda x: [str(y).lower() for y in x]).apply(clean_movie_genres)

In [27]:
df_movies_summaries['Release year'] = pd.to_datetime(df_movies_summaries['Release date'], format='%Y-%m-%d', errors='coerce').dt.year

df_movies_summaries['Release year'] = df_movies_summaries['Release year'].fillna(pd.to_datetime(df_movies_summaries['Release date'], format='%Y', errors='coerce').dt.year)

In [20]:
df_movies_summaries["Languages"] = df_movies_summaries["Languages"].apply(extract_tuples_values)

In [29]:
df_movies_summaries["Countries"] = df_movies_summaries["Countries"].apply(extract_tuples_values)

In [30]:
df_movies_summaries

,Movie name,Release date,Box office revenue,Runtime,Languages,Countries,Genres,Summary,Release year
0,White Of The Eye,1987,NaN,110.0,[English Language],[United Kingdom],"[biography, crime, drama, mystery]",A series of murders of rich young women throug...,1987.0
1,A Woman in Flames,1983,NaN,106.0,[German Language],[Germany],"[crime, drama, fiction]","Eva, an upper class housewife, becomes frustra...",1983.0
2,The Sorcerer's Apprentice,2002,NaN,86.0,[English Language],[South Africa],"[erotic, psychological, thriller]","Every hundred years, the evil Morgana returns...",2002.0
3,Little city,1997-04-04,NaN,93.0,[English Language],[United States of America],[drama],"Adam, a San Francisco-based artist who works a...",1997.0
4,Henry V,1989-11-08,10161099.0,137.0,[English Language],[United Kingdom],"[black and white, comedy, indie, short, silent]",{{Plot|dateAct 1Act 2Act 3Act 4Act 5 Finally n...,1989.0
...,...,...,...,...,...,...,...,...,...
42197,The Ghost Train,1941-05-03,NaN,82.0,[English Language],[United Kingdom],"[action, thriller]",{{plot}} The film opens with a Great Western e...,1941.0
42198,Mermaids: The Body Found,2011-03-19,NaN,120.0,[English Language],[United States of America],"[action, comedy, musical, romance, western]",Two former National Oceanic Atmospheric Admini...,2011.0
42199,Knuckle,2011-01-21,NaN,96.0,[English Language],"[Ireland, United Kingdom]","[drama, romance, war]",{{No plot}} This film follows 12 years in the ...,2011.0
42200,The Super Dimension Fortress Macross II: Lover...,1992-05-21,NaN,150.0,[Japanese Language],[Japan],[documentary],"The story takes place in the year 2092,The Sup...",1992.0


In [41]:
df_tmdb['clean_title'] = df_tmdb['title'].str.lower().str.strip()
df_movies_summaries['clean_title'] = df_movies_summaries['Movie name'].str.lower().str.strip()

In [36]:
df_tmdb['release_year'] = pd.to_datetime(df_tmdb['release_date'], format='%Y-%m-%d', errors='coerce').dt.year

In [44]:
df_cmu_tmdb = pd.merge(df_movies_summaries, df_tmdb, left_on=['clean_title', 'Release year'], right_on = ['clean_title','release_year'], how='inner')

In [80]:
df_cmu_tmdb['genres2'] = df_cmu_tmdb['genres'].str.split(",").tolist()

In [81]:
df_cmu_tmdb['genres2'] = df_cmu_tmdb['genres2'].apply(lambda x: [str(y).lower().strip() for y in x] if isinstance(x, list) else [])

In [85]:
df_cmu_tmdb['all_genres'] = df_cmu_tmdb.apply(lambda row: list(set(row['Genres'] + row['genres2'])), axis=1)

In [170]:
df_cmu_tmdb[df_cmu_tmdb['Summary'].apply(lambda x: ("computer" in x.lower())if isinstance(x, str) else False)][['Summary', 'keywords', 'all_genres']]

,Summary,keywords,all_genres
84,Computer experts around the world strive towar...,NaN,"[world cinema, documentary, drama]"
86,"Omni Consumer Products , on the verge of bankr...","police, cyborg, dystopia, sequel, cyberpunk, p...","[action, science fiction, crime, thriller, adv..."
94,"Six introverted individuals, bored with their ...",NaN,"[romance, romantic, comedy, drama, world cinema]"
101,The USS Charleston in On the Beach is a 688i ...,"australia, based on novel or book, submarine, ...","[science fiction, comedy, surrealism, avant ga..."
320,"Following the events of Puppet Master 4, Rick ...","hotel, puppet, sequel, evil doll, demon, egypt...","[fantasy, science fiction, short, animation, f..."
...,...,...,...
30227,"{{plot}} In an underground cave, Tolbiac wak...","amnesia, cave, plant","[action, science fiction, crime, fiction, film..."
30306,Geena is a Bollywood fanatic from a respectab...,"london, england, interracial relationship, sha...","[romance, japanese, music, drama, world cinema..."
30339,U.S. energy giant Connex is losing control of ...,"petrol, cia, middle east, capitalism, globaliz...","[thriller, drama]"
30383,Frank Martin is a highly skilled driver known...,"martial arts, transporter, france, human traff...","[action, romance, romantic, black and white, s..."


Keywords: 
to classify: espionage, cia
- World War: world war, trench, germany, nazi
- Vietnam War: vietnam, guerilla, 
- Cold War: russia, 
- War on terror: soviet union, ussr
- ...

In [123]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(all_summaries)

# Convert the TF-IDF matrix to a DataFrame for easier analysis
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Calculate the average TF-IDF score for each term across documents
mean_tfidf = tfidf_df.mean().sort_values(ascending=False)

# Get top N words with highest mean TF-IDF scores
top_n = 5
top_words = mean_tfidf.head(top_n)


In [158]:
df_cmu_tmdb[df_cmu_tmdb['clean_title'] == 'saving private ryan']['Summary']

9139    On the morning of June 6, 1944, the beginning ...
Name: Summary, dtype: object